# Geometric decision-making in a neural spin model

**DRAFT for review.** This notebook uses MuMoT to reproduce, in a simplified mean-field and stochastic form, the core mechanism of the neural *spin model* of spatial decision-making in

* V. H. Sridhar, L. Li, D. Gorbonos, M. Nagy, B. R. Schell, T. Sorochkin, N. S. Gov and I. D. Couzin (2021) *The geometry of decision-making in individuals and collectives*. PNAS **118**(50): e2102157118. [doi:10.1073/pnas.2102157118](https://doi.org/10.1073/pnas.2102157118)
* D. Gorbonos, N. S. Gov and I. D. Couzin (2024) *Geometrical structure of bifurcations during spatial decision-making*. PRX Life **2**: 013008. [doi:10.1103/PRXLife.2.013008](https://doi.org/10.1103/PRXLife.2.013008)

In these papers an animal moving towards several targets represents each target direction with a group of neurons, modelled as Ising-like spins (as in a Hopfield network). Neurons coding for the same direction excite each other; the interaction between groups depends on the angle between the target directions. When the targets are close together in angle the animal *compromises*, moving in the average direction; beyond a critical angle it abruptly *decides*, moving towards one target. With more targets this happens repeatedly, breaking a multi-choice decision into a series of binary ones.

MuMoT cannot simulate the embodied agent (its movement changes the angles to the targets over time, which drives the sequence of bifurcations in the papers). What it can do is analyse the neural decision dynamics at **fixed** target angles: steady states, their stability, the bifurcation from compromise to decision, and the effect of finite numbers of neurons (noise).

> **Notes for the reviewer** are marked like this. In particular the spin-model details below were written without access to the papers' full text and must be checked against them.

In [ ]:
import mumot
import numpy as np
import matplotlib.pyplot as plt

## From the spin model to a MuMoT model

In the spin model each neuron $i$ is a spin $\sigma_i \in \{0, 1\}$ (inactive/active), assigned to one target direction. The energy is of Hopfield form

$$H = -\frac{k}{N} \sum_{i \neq j} J_{ij}\, \sigma_i \sigma_j, \qquad J_{ij} = \cos\left(\pi \left(\frac{|\theta_{ij}|}{\pi}\right)^{\nu}\right),$$

where $\theta_{ij}$ is the angle between the target directions coded by neurons $i$ and $j$ (so $J_{ij} = 1$ within a group), and $\nu$ controls how the brain represents angles ($\nu = 1$: the plain cosine). Spins flip stochastically at a 'temperature' (neural noise), and the agent moves in the average direction of the targets coded by active neurons.

> **Reviewer:** check the exact Hamiltonian, normalisation ($k$, $N$, any constraint on total activity), the dynamics (Glauber/Metropolis) and the definition of $\nu$ against the papers.

To express this in MuMoT we treat the neurons as a population with three states: active for target 1 ($A$), active for target 2 ($B$) and inactive ($U$), and replace the spin-flip dynamics by mass-action reactions with the same sign structure as the couplings:

| spin model | reaction | rate |
|---|---|---|
| spontaneous activation (noise) | $U \to A$, $U \to B$ | $g$ |
| spontaneous deactivation | $A \to U$, $B \to U$ | $a$ |
| excitation within a group ($J = 1$) | $A + U \to A + A$, $B + U \to B + B$ | $r$ |
| excitation between groups ($J_{AB} > 0$, acute angles) | $B + U \to B + A$, $A + U \to A + B$ | $c_E = r \max(J_{AB}, 0)$ |
| inhibition between groups ($J_{AB} < 0$, obtuse angles) | $A + B \to A + U$, $A + B \to B + U$ | $c_I = r \max(-J_{AB}, 0)$ |

With $c_E = 0$ this is exactly the cross-inhibition ('stop-signal') model of house-hunting honeybees analysed earlier in the user manual, here with equal option values: the neural spin model and the collective decision model share the same decision mechanism.

In [ ]:
%%model
$
U -> A : g
U -> B : g
A -> U : a
B -> U : a
A + U -> A + A : r
B + U -> B + B : r
B + U -> B + A : c_E
A + U -> A + B : c_E
A + B -> A + U : c_I
A + B -> B + U : c_I
$

In [ ]:
model1 = mumot.parseModel(In[-2])

In [ ]:
model1.visualise()

The total number of neurons is fixed, so we eliminate the inactive state, $U = N - A - B$, and look at the mean-field equations for the fractions of active neurons $\Phi_A$ and $\Phi_B$:

In [ ]:
model2 = model1.substitute('U = N - A - B')
model2.showODEs()

### Couplings from the geometry

A small helper converts the angle between the two targets into the coupling rates, for a given angle representation $\nu$ and within-group excitation $r$.

In [ ]:
def coupling(theta_deg, nu=1.0, r=4.0):
    """Between-group coupling J and the rates (c_E, c_I) for targets theta_deg degrees apart."""
    theta = np.radians(theta_deg)
    J = np.cos(np.pi * (theta / np.pi)**nu)
    return J, r * max(J, 0.0), r * max(-J, 0.0)

for theta in (30, 60, 90, 120, 150):
    J, c_E, c_I = coupling(theta)
    print(f"theta = {theta:3d} deg: J = {J:+.2f}, c_E = {c_E:.2f}, c_I = {c_I:.2f}")

## Compromise and decision

We use $g = 0.1$ (noise), $a = 1$ and $r = 4$ throughout (relative to $a$). First, two targets $60°$ apart: the groups excite each other, both stay equally active and the agent heads between the targets (compromise).

In [ ]:
J, c_E, c_I = coupling(60)
compromise = model2.integrate(params=[('g', 0.1), ('a', 1), ('r', 4), ('c_E', c_E), ('c_I', c_I)],
                              initWidgets={'initialState': {'A': [0.12, 0, 1, 0.01], 'B': [0.1, 0, 1, 0.01],
                                                            'U': [0.78, 0, 1, 0.01]},
                                           'maxTime': [10, 1, 50, 1]}, plotProportions=True)

Now $120°$ apart: the groups inhibit each other and a slight initial advantage for target 1 grows until one group dominates (decision).

In [ ]:
J, c_E, c_I = coupling(120)
decision = model2.integrate(params=[('g', 0.1), ('a', 1), ('r', 4), ('c_E', c_E), ('c_I', c_I)],
                            initWidgets={'initialState': {'A': [0.12, 0, 1, 0.01], 'B': [0.1, 0, 1, 0.01],
                                                          'U': [0.78, 0, 1, 0.01]},
                                         'maxTime': [10, 1, 50, 1]}, plotProportions=True)

The phase planes show why: at $60°$ there is a single stable state on the diagonal $\Phi_A = \Phi_B$; at $120°$ that state has become unstable (a saddle), flanked by two stable decision states.

In [ ]:
J, c_E, c_I = coupling(60)
stream60 = model2.stream('A', 'B', showFixedPoints=True,
                         params=[('g', 0.1), ('a', 1), ('r', 4), ('c_E', c_E), ('c_I', c_I)])

In [ ]:
J, c_E, c_I = coupling(120)
stream120 = model2.stream('A', 'B', showFixedPoints=True,
                          params=[('g', 0.1), ('a', 1), ('r', 4), ('c_E', c_E), ('c_I', c_I)])

## The bifurcation

For obtuse angles ($c_E = 0$) the inhibition $c_I = -rJ$ grows with the angle. Continuing the steady states in $c_I$ shows a pitchfork bifurcation: the compromise state loses stability at a critical inhibition and two symmetric decision branches appear.

In [ ]:
bifurcation1 = model2.bifurcation('c_I', 'A-B', params=[('c_E', 0)],
                                  initWidgets={'g': [0.1, 0, 1, 0.05], 'a': [1, 0.1, 2, 0.1], 'r': [4, 0, 10, 0.5],
                                               'initBifParam': [0.1, 0.05, 4, 0.05]},
                                  choose_xrange=[0, 4], xlab=r'inhibition $c_I$')

### From critical coupling to critical angle

The same mean-field equations, evaluated numerically from MuMoT's ODEs, give the agent's heading as a function of the angle between the targets. Its direction of motion is the activity-weighted average of the two target directions.

In [ ]:
from scipy.integrate import solve_ivp
import sympy

odes = model2.getODEs()
A, B = sorted(odes, key=str)
params = sorted(set().union(*(eq.free_symbols for eq in odes.values())) - {A, B}, key=str)
rhs = sympy.lambdify([A, B] + params, [odes[A], odes[B]], 'numpy')
names = [str(p).replace('{', '').replace('}', '') for p in params]   # e.g. 'c_E', 'N'

def stable_states(theta_deg, nu=1.0, g=0.1, a=1.0, r=4.0, starts=((0.3, 0.2), (0.2, 0.3))):
    J, c_E, c_I = coupling(theta_deg, nu, r)
    values = dict(g=g, a=a, r=r, c_E=c_E, c_I=c_I, N=1)
    p = [values[n] for n in names]
    f = lambda t, y: rhs(*y, *p)
    return {tuple(np.round(solve_ivp(f, (0, 500), y0, rtol=1e-9).y[:, -1], 4)) for y0 in starts}

def headings(theta_deg, **kw):
    half = np.radians(theta_deg) / 2   # targets at -half and +half
    return sorted({round(float(np.degrees(np.arctan2(phiB * np.sin(half) - phiA * np.sin(half),
                                                         phiA * np.cos(half) + phiB * np.cos(half)))), 2)
                   for phiA, phiB in stable_states(theta_deg, **kw)})

angles = np.arange(2, 180, 2)
fig, ax = plt.subplots(figsize=(6, 4))
for nu, colour in [(1.0, 'C0'), (0.5, 'C1')]:
    for theta in angles:
        for h in headings(theta, nu=nu):
            ax.plot(theta, h, '.', color=colour, label=rf'$\nu = {nu}$' if theta == angles[0] else None)
ax.plot(angles, angles / 2, 'k:', lw=0.8, label='target directions')
ax.plot(angles, -angles / 2, 'k:', lw=0.8)
ax.set_xlabel('angle between targets (degrees)')
ax.set_ylabel('heading (degrees)')
ax.legend()
fig.tight_layout()
plt.show()

Below the critical angle the agent heads straight between the targets (heading $0°$, compromise); above it, it heads (almost) straight for one of them. With $\nu < 1$ the brain's representation of angles is compressed towards $180°$ and the decision happens at a smaller geometric angle.

> **Reviewer:** compare with the critical angles in the papers (for the values of $\nu$ they use, and how the critical angle depends on noise). The notebook's $g$ plays the role of neural noise and $r$ of the coupling strength $k$ (inverse temperature).

In [ ]:
def critical_angle(nu=1.0, g=0.1, r=4.0):
    """Smallest angle (to 0.5 degree) at which the agent no longer heads between the targets."""
    for theta in np.arange(1, 180, 0.5):
        if max(abs(h) for h in headings(theta, nu=nu, g=g, r=r)) > 1:
            return theta
    return None

for g in (0.05, 0.1, 0.2, 0.4):
    print(f"noise g = {g:4}: critical angle {critical_angle(g=g)} deg (nu = 1), {critical_angle(nu=0.5, g=g)} deg (nu = 0.5)")

## Finite numbers of neurons

With a finite number of neurons $N$ the decision is stochastic. Above the critical angle, repeated simulations decide for either target at random; near it, fluctuations make the decision slow and noisy. Use the *System size* slider to see the effect of $N$.

In [ ]:
J, c_E, c_I = coupling(120)
ssa1 = model2.SSA(params=[('g', 0.1), ('a', 1), ('r', 4), ('c_E', c_E), ('c_I', c_I)],
                  initWidgets={'initialState': {'A': [0, 0, 1, 0.01], 'B': [0, 0, 1, 0.01], 'U': [1, 0, 1, 0.01]},
                               'maxTime': [20, 1, 100, 1], 'systemSize': [50, 10, 500, 10], 'runs': [5, 1, 20, 1],
                               'aggregateResults': False}, plotProportions=True)

## Three targets (exploratory)

> **Reviewer:** this section is the least certain. In the papers the sequence of bifurcations with three or more targets arises as the moving agent's angles to the targets change. Here the angles are fixed, so only snapshots of that process can be studied.

With three targets at $-\theta$, $0$ and $+\theta$ (neurons $A$, $B$, $C$), each pair interacts according to its own angle.

In [ ]:
%%model
$
U -> A : g
U -> B : g
U -> C : g
A -> U : a
B -> U : a
C -> U : a
A + U -> A + A : r
B + U -> B + B : r
C + U -> C + C : r
B + U -> B + A : e_{AB}
A + U -> A + B : e_{AB}
C + U -> C + B : e_{BC}
B + U -> B + C : e_{BC}
C + U -> C + A : e_{AC}
A + U -> A + C : e_{AC}
A + B -> A + U : i_{AB}
A + B -> B + U : i_{AB}
B + C -> B + U : i_{BC}
B + C -> C + U : i_{BC}
A + C -> A + U : i_{AC}
A + C -> C + U : i_{AC}
$

In [ ]:
model3 = mumot.parseModel(In[-2]).substitute('U = N - A - B - C')

In [ ]:
def three_target_params(theta_deg, nu=1.0, r=4.0):
    """Partial-controller parameters for targets at -theta, 0, +theta."""
    pairs = {'AB': theta_deg, 'BC': theta_deg, 'AC': min(2 * theta_deg, 360 - 2 * theta_deg)}
    p = [('g', 0.1), ('a', 1), ('r', r)]
    for pair, angle in pairs.items():
        J, c_E, c_I = coupling(angle, nu, r)
        p += [('e_{' + pair + '}', c_E), ('i_{' + pair + '}', c_I)]
    return p

three60 = model3.integrate(params=three_target_params(60),
                           initWidgets={'initialState': {'A': [0.11, 0, 1, 0.01], 'B': [0.1, 0, 1, 0.01],
                                                         'C': [0.1, 0, 1, 0.01], 'U': [0.69, 0, 1, 0.01]},
                                        'maxTime': [20, 1, 50, 1]}, plotProportions=True)

In [ ]:
three130 = model3.integrate(params=three_target_params(130),
                            initWidgets={'initialState': {'A': [0.11, 0, 1, 0.01], 'B': [0.1, 0, 1, 0.01],
                                                          'C': [0.1, 0, 1, 0.01], 'U': [0.69, 0, 1, 0.01]},
                                         'maxTime': [20, 1, 50, 1]}, plotProportions=True)

At $60°$ all three groups stay active: a compromise heading towards the middle target. At $130°$ the decision unfolds as a sequence of binary decisions in time, as in the papers: first the middle target is excluded (the outer groups both inhibit it), then the two outer groups compete until one wins. Use the sliders to explore other angles and noise levels.

## Open questions for review

1. Check the spin-model description (Hamiltonian, $J_{ij}$, $\nu$, noise, dynamics) and the citations against the papers.
2. Is the split of $J$ into excitation ($J > 0$) and inhibition ($J < 0$) reactions the right mass-action analogue of the spin dynamics? An alternative is $c_E = r(1 + J)/2$, $c_I = r(1 - J)/2$, which keeps the net coupling $rJ$ but adds both terms at every angle (it gives a critical angle of about $115°$ for $\nu = 1$ instead of about $95°$, and different three-target behaviour).
3. Which quantitative results from the papers (critical angles for given $\nu$ and noise) should the notebook aim to match, and with which parameter values?
4. Should the three-target section reproduce a specific figure, e.g. by choosing angles that correspond to points along an agent's trajectory?